In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score,KFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import ElasticNet,Ridge
from sklearn.preprocessing import StandardScaler
import optuna
from sklearn.ensemble import StackingRegressor
import joblib

In [5]:
X_train = pd.read_parquet("../data/processed/train_X_fe.parquet")
y_train = pd.read_parquet("../data/processed/train_y.parquet").squeeze()

In [6]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)  

# Random Forest

In [7]:
def objective_rf(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 30),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    scores = cross_val_score(
        model,
        X_train, y_train,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )

    rmse = -scores.mean()

    return rmse

In [8]:
study_rf = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_rf.optimize(objective_rf, n_trials=100, show_progress_bar=True)

print("Best params:", study_rf.best_params)
print("Best CV RMSE (on log-scale):", study_rf.best_value)

[I 2026-03-02 14:17:48,333] A new study created in memory with name: no-name-9194e14d-dbd2-4506-a67b-2e72370ebdae


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-02 14:17:54,758] Trial 0 finished with value: 0.14907493031015323 and parameters: {'n_estimators': 500, 'max_depth': 29, 'min_samples_split': 37, 'min_samples_leaf': 18, 'max_features': 0.40921304830970556, 'bootstrap': True}. Best is trial 0 with value: 0.14907493031015323.
[I 2026-03-02 14:18:10,583] Trial 1 finished with value: 0.15019068990025042 and parameters: {'n_estimators': 893, 'max_depth': 19, 'min_samples_split': 36, 'min_samples_leaf': 1, 'max_features': 0.978936896513396, 'bootstrap': True}. Best is trial 0 with value: 0.14907493031015323.
[I 2026-03-02 14:18:15,509] Trial 2 finished with value: 0.14927010897631848 and parameters: {'n_estimators': 345, 'max_depth': 8, 'min_samples_split': 16, 'min_samples_leaf': 16, 'max_features': 0.602361513049481, 'bootstrap': False}. Best is trial 0 with value: 0.14907493031015323.
[I 2026-03-02 14:18:22,246] Trial 3 finished with value: 0.15981671890692542 and parameters: {'n_estimators': 311, 'max_depth': 11, 'min_samples

In [9]:
joblib.dump(study_rf, "../models/study_rf.pkl")

['../models/study_rf.pkl']

In [10]:
rf_best = RandomForestRegressor(**study_rf.best_params)
rf_best.fit(X_train, y_train)
joblib.dump(rf_best, "../models/rf_model.pkl")

['../models/rf_model.pkl']

# Ridge Regression

In [11]:
def objective_ridge(trial):

    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 100, log=True),
        "random_state": 42
    }

    model = Ridge(**params)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )

    rmse = -scores.mean()


    return rmse

In [12]:
study_ridge = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_ridge.optimize(objective_ridge, n_trials=100, show_progress_bar=True)

print("Best Ridge params:", study_ridge.best_params)
print("Best CV RMSE (on log-scale):", study_ridge.best_value)

[I 2026-03-02 14:36:07,357] A new study created in memory with name: no-name-1b6407ea-2ebe-4151-baf2-a496a56c1657


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-02 14:36:07,583] Trial 0 finished with value: 0.11872217502947557 and parameters: {'alpha': 0.004185822729546971}. Best is trial 0 with value: 0.11872217502947557.
[I 2026-03-02 14:36:07,795] Trial 1 finished with value: 0.11412764059310032 and parameters: {'alpha': 45.18560951024108}. Best is trial 1 with value: 0.11412764059310032.
[I 2026-03-02 14:36:08,004] Trial 2 finished with value: 0.11819774395882873 and parameters: {'alpha': 1.330324510152291}. Best is trial 1 with value: 0.11412764059310032.
[I 2026-03-02 14:36:08,224] Trial 3 finished with value: 0.11865629958708857 and parameters: {'alpha': 0.1550991398759431}. Best is trial 1 with value: 0.11412764059310032.
[I 2026-03-02 14:36:08,446] Trial 4 finished with value: 0.11872397392113103 and parameters: {'alpha': 0.000123631882770522}. Best is trial 1 with value: 0.11412764059310032.
[I 2026-03-02 14:36:08,678] Trial 5 finished with value: 0.11872397394240489 and parameters: {'alpha': 0.0001235838277230692}. Best i

In [13]:
joblib.dump(study_ridge, "../models/study_ridge.pkl")

['../models/study_ridge.pkl']

In [14]:
ridge_best = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(**study_ridge.best_params))
])
ridge_best.fit(X_train, y_train)
joblib.dump(ridge_best, "../models/ridge_model.pkl")

['../models/ridge_model.pkl']

# XGBoost

In [15]:
def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 1),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),

        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBRegressor(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )

    rmse = -scores.mean()

    return rmse

In [16]:
study_xgb = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_xgb.optimize(objective_xgb, n_trials=100, show_progress_bar=True)

print("Best XGB params:", study_xgb.best_params)
print("Best CV RMSE (on log-scale):", study_xgb.best_value)

[I 2026-03-02 14:36:28,771] A new study created in memory with name: no-name-0b593feb-829a-46ae-94c7-48d00ce3825f


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-02 14:36:31,689] Trial 0 finished with value: 0.12355355819372063 and parameters: {'n_estimators': 1062, 'max_depth': 8, 'learning_rate': 0.05395030966670229, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.05808361216819946, 'reg_alpha': 0.08499808989182997, 'reg_lambda': 1.5930522616241019}. Best is trial 0 with value: 0.12355355819372063.
[I 2026-03-02 14:36:34,750] Trial 1 finished with value: 0.12797940622998277 and parameters: {'n_estimators': 1562, 'max_depth': 3, 'learning_rate': 0.09330606024425668, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 2, 'gamma': 0.18340450985343382, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 1.1207606211860568}. Best is trial 0 with value: 0.12355355819372063.
[I 2026-03-02 14:36:37,690] Trial 2 finished with value: 0.13410440851655686 and parameters: {'n_estimators': 1148, 'max_depth': 4, 'learning_rate': 0.04091220574443785, 

In [17]:
joblib.dump(study_xgb,"../models/study_xgb.pkl")

['../models/study_xgb.pkl']

In [18]:
xgb_best = XGBRegressor(**study_xgb.best_params)
xgb_best.fit(X_train, y_train)
joblib.dump(xgb_best, "../models/xgb_model.pkl")

['../models/xgb_model.pkl']

# LightGBM

In [19]:
def objective_lgb(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 800, 2500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "max_depth": trial.suggest_int("max_depth", 3, 10),

        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),

        "random_state": 42,
        "n_jobs": -1
    }

    model = LGBMRegressor(**params)

    scores = cross_val_score(
        model, X_train, y_train,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )

    rmse = -scores.mean()

    return rmse

In [20]:
study_lgb = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study_lgb.optimize(objective_lgb, n_trials=100, show_progress_bar=True)

print("Best LGB params:", study_lgb.best_params)
print("Best CV RMSE (on log-scale):", study_lgb.best_value)

[I 2026-03-02 14:40:47,810] A new study created in memory with name: no-name-93de79b1-2308-4827-b162-4e8bb88b5277


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-02 14:40:57,848] Trial 0 finished with value: 0.12388808790642822 and parameters: {'n_estimators': 1437, 'learning_rate': 0.046187109390049115, 'num_leaves': 51, 'max_depth': 7, 'min_child_samples': 24, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_alpha': 0.08499808989182997, 'reg_lambda': 1.5930522616241019}. Best is trial 0 with value: 0.12388808790642822.
[I 2026-03-02 14:41:09,573] Trial 1 finished with value: 0.12292002942086307 and parameters: {'n_estimators': 2004, 'learning_rate': 0.010336843570697411, 'num_leaves': 63, 'max_depth': 9, 'min_child_samples': 29, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 1.1207606211860568}. Best is trial 1 with value: 0.12292002942086307.
[I 2026-03-02 14:41:13,196] Trial 2 finished with value: 0.12483078559991885 and parameters: {'n_estimators': 1534, 'learning_rate': 0.01597939871741036, 'num_leaves': 45, 'max_depth': 4, '

In [21]:
joblib.dump(study_lgb,"../models/study_lgb.pkl")

['../models/study_lgb.pkl']

In [22]:
lgb_best = LGBMRegressor(**study_lgb.best_params)
lgb_best.fit(X_train, y_train)
joblib.dump(lgb_best, "../models/lgb_model.pkl")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003002 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2444
[LightGBM] [Info] Number of data points in the train set: 1238, number of used features: 151
[LightGBM] [Info] Start training from score 12.022526
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

['../models/lgb_model.pkl']

# Stacking

In [23]:
lgb_model   = LGBMRegressor(**study_lgb.best_params)
xgb_model   = XGBRegressor(**study_xgb.best_params)
rf_model    = RandomForestRegressor(**study_rf.best_params)
ridge_model = Pipeline([("scaler", StandardScaler()),("model", Ridge(**study_ridge.best_params))])

stack = StackingRegressor(
    estimators=[
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('rf', rf_model),
        ('ridge', ridge_model)
    ],
    final_estimator=Ridge(),
    cv=5
)

stack.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2444
[LightGBM] [Info] Number of data points in the train set: 1238, number of used features: 151
[LightGBM] [Info] Start training from score 12.022526
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

StackingRegressor(cv=5,
                  estimators=[('lgb',
                               LGBMRegressor(colsample_bytree=0.6425209041845269,
                                             learning_rate=0.034129126638240456,
                                             max_depth=3, min_child_samples=14,
                                             n_estimators=1216, num_leaves=25,
                                             reg_alpha=6.228001168355196e-05,
                                             reg_lambda=0.7496206985771338,
                                             subsample=0.9846269409125783)),
                              ('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            c...
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=780, n_jobs=None,
                                            num_parallel_tree=None, ...)),
                              ('rf',
                               RandomForestRegressor(bootstrap=False,
                                                     max_depth=13,
                                                     max_features=0.37593826431306193,
                                                     min_samples_leaf=3,
                                                     min_samples_split=7,
                                                     n_estimators=709)),
                              ('ridge',
                               Pipeline(steps=[('scaler', StandardScaler()),
                                               ('model',
                                                Ridge(alpha=76.20540890505636))]))],
                  final_estimator=Ridge())

In [24]:
scores = cross_val_score(
    stack,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

print("CV RMSE (on log-scale):", -scores.mean())

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002856 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2305
[LightGBM] [Info] Number of data points in the train set: 990, number of used features: 146
[LightGBM] [Info] Start training from score 12.027392
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [25]:
X_holdout = pd.read_parquet("../data/processed/holdout_X.parquet")
y_holdout = pd.read_parquet("../data/processed/holdout_y.parquet").squeeze()

preds = stack.predict(X_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_holdout, preds))
print("Holdout CV RMSE (on log-scale):", holdout_rmse)

Holdout CV RMSE (on log-scale): 0.1156916071118393


In [26]:
joblib.dump(stack, "../models/stack_model.pkl")

['../models/stack_model.pkl']